# Neural metadata embeddings

This notebook consumes stream, channel, category, follow, and VOD snapshots from `00_get_data.ipynb`. It trains neural token embeddings from metadata that co-occurs on live streams, creates vectors for streams, categories, and channels, and ranks live streams by cosine similarity to the user's observed preferences.

This is an unsupervised warm start. The next iteration should train with watch, click, follow, and skip labels.

## Load exported metadata

In [1]:
import random
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
notebook_dir = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
data_dir = notebook_dir / "data"

def load_jsonl(name: str) -> pd.DataFrame:
    path = data_dir / f"{name}.jsonl"
    return pd.read_json(path, lines=True) if path.exists() else pd.DataFrame()

streams_df = load_jsonl("live_stream_features")
channels_df = load_jsonl("channel_metadata")
categories_df = load_jsonl("category_metadata")
followed_channels_df = load_jsonl("channels")
vods_df = load_jsonl("vods")
if streams_df.empty:
    raise FileNotFoundError("Run 00_get_data.ipynb first to export live_stream_features.jsonl.")
print({"streams": len(streams_df), "channel_metadata": len(channels_df), "category_metadata": len(categories_df), "followed_channels": len(followed_channels_df), "vods": len(vods_df)})

{'streams': 5000, 'channel_metadata': 4978, 'category_metadata': 665, 'followed_channels': 0, 'vods': 0}


## Build metadata token sequences

Each stream becomes a short sequence containing category, channel, language, tags, and channel metadata tokens. Tokens that occur together provide positive training examples for the embedding model.

In [2]:
channel_lookup = ({str(key): value for key, value in channels_df.set_index("broadcaster_id").to_dict("index").items()} if not channels_df.empty and "broadcaster_id" in channels_df else {})
category_lookup = ({str(key): value for key, value in categories_df.set_index("id").to_dict("index").items()} if not categories_df.empty and "id" in categories_df else {})

def clean_token(value: object) -> str:
    return str(value).strip().lower().replace(" ", "_")

def stream_tokens(row: pd.Series) -> list[str]:
    tokens = []
    if pd.notna(row.get("game_id")):
        tokens.append(f"category_id:{clean_token(row.get('game_id'))}")
    category_name = category_lookup.get(str(row.get("game_id")), {}).get("name", row.get("game_name"))
    if pd.notna(category_name):
        tokens.append(f"category:{clean_token(category_name)}")
    if pd.notna(row.get("user_id")):
        tokens.append(f"channel:{clean_token(row.get('user_id'))}")
    if pd.notna(row.get("language")):
        tokens.append(f"language:{clean_token(row.get('language'))}")
    tags = row.get("tags", [])
    if isinstance(tags, list):
        tokens.extend(f"tag:{clean_token(tag)}" for tag in tags if pd.notna(tag))
    channel = channel_lookup.get(str(row.get("user_id")), {})
    channel_tags = channel.get("tags", [])
    if isinstance(channel_tags, list):
        tokens.extend(f"channel_tag:{clean_token(tag)}" for tag in channel_tags if pd.notna(tag))
    return list(dict.fromkeys(tokens))

streams_df["metadata_tokens"] = streams_df.apply(stream_tokens, axis=1)
streams_df = streams_df[streams_df["metadata_tokens"].map(len) >= 2].reset_index(drop=True)
vocabulary = sorted({token for tokens in streams_df["metadata_tokens"] for token in tokens})
token_to_id = {token: index for index, token in enumerate(vocabulary)}
print(f"Built {len(vocabulary)} metadata tokens across {len(streams_df)} streams")

Built 22906 metadata tokens across 5000 streams


## Train neural token embeddings

A skip-gram objective pulls tokens from the same stream together and pushes randomly sampled unrelated tokens apart. The learned token vectors are then averaged into stream, category, and channel vectors.

In [3]:
embedding_size = 32
positive_pairs = []
for tokens in streams_df['metadata_tokens']:
    ids = [token_to_id[token] for token in tokens]
    positive_pairs.extend(combinations(ids, 2))
    positive_pairs.extend((right, left) for left, right in combinations(ids, 2))
if len(positive_pairs) > 50000:
    positive_pairs = random.sample(positive_pairs, 50000)

class SkipGram(nn.Module):
    def __init__(self, vocabulary_size: int, dimensions: int):
        super().__init__()
        self.input_embeddings = nn.Embedding(vocabulary_size, dimensions)
        self.context_embeddings = nn.Embedding(vocabulary_size, dimensions)

    def forward(self, center: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        return (self.input_embeddings(center) * self.context_embeddings(context)).sum(dim=1)

embedding_model = SkipGram(len(vocabulary), embedding_size)
optimizer = torch.optim.Adam(embedding_model.parameters(), lr=0.01)
loss_function = nn.BCEWithLogitsLoss()
pair_tensor = torch.tensor(positive_pairs, dtype=torch.long)
batch_size = 512

for epoch in range(40):
    permutation = torch.randperm(len(pair_tensor))
    epoch_loss = 0.0
    for start in range(0, len(pair_tensor), batch_size):
        batch = pair_tensor[permutation[start:start + batch_size]]
        negatives = torch.randint(0, len(vocabulary), (len(batch),), dtype=torch.long)
        centers = torch.cat([batch[:, 0], batch[:, 0]])
        contexts = torch.cat([batch[:, 1], negatives])
        labels = torch.cat([torch.ones(len(batch)), torch.zeros(len(batch))])
        loss = loss_function(embedding_model(centers, contexts), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    if epoch in {0, 9, 19, 39}:
        print(f'epoch={epoch + 1}, loss={epoch_loss:.4f}')

token_vectors = embedding_model.input_embeddings.weight.detach().numpy()
token_vectors = token_vectors / np.maximum(np.linalg.norm(token_vectors, axis=1, keepdims=True), 1e-12)

epoch=1, loss=219.2776


epoch=10, loss=82.8002


epoch=20, loss=51.4355


epoch=40, loss=13.2283


## Create stream, category, channel, and user vectors

In [4]:
def average_vector(tokens: list[str]) -> np.ndarray:
    ids = [token_to_id[token] for token in tokens if token in token_to_id]
    if not ids:
        return np.zeros(embedding_size)
    vector = token_vectors[ids].mean(axis=0)
    return vector / max(np.linalg.norm(vector), 1e-12)

stream_vectors = np.vstack(streams_df['metadata_tokens'].map(average_vector).to_numpy())

def vectors_for_prefix(prefix: str) -> pd.DataFrame:
    rows = [(token[len(prefix):], token_vectors[index].tolist()) for token, index in token_to_id.items() if token.startswith(prefix)]
    return pd.DataFrame(rows, columns=['key', 'embedding'])

category_embeddings_df = vectors_for_prefix('category:')
channel_embeddings_df = vectors_for_prefix('channel:')

preference_tokens = []
if not followed_channels_df.empty and 'broadcaster_id' in followed_channels_df:
    preference_tokens.extend(f'channel:{clean_token(value)}' for value in followed_channels_df['broadcaster_id'].dropna())
if not vods_df.empty and 'game_id' in vods_df:
    preference_tokens.extend(f'category_id:{clean_token(value)}' for value in vods_df['game_id'].dropna())
user_vector = average_vector(preference_tokens) if preference_tokens else np.zeros(embedding_size)
print({'category_embeddings': len(category_embeddings_df), 'channel_embeddings': len(channel_embeddings_df), 'preference_tokens': len(preference_tokens)})

{'category_embeddings': 660, 'channel_embeddings': 4978, 'preference_tokens': 0}


## Cosine similarity and recommendations

The metadata score is cosine similarity between each stream vector and the user's preference vector. If no followed/VOD context exists, the notebook reports zero similarity rather than inventing a preference vector.

In [5]:
if np.linalg.norm(user_vector) == 0:
    streams_df['metadata_similarity'] = 0.0
else:
    streams_df['metadata_similarity'] = stream_vectors @ user_vector

streams_df['viewer_signal'] = streams_df['viewer_log1p'].rank(pct=True)
streams_df['embedding_score'] = 0.70 * streams_df['metadata_similarity'] + 0.30 * streams_df['viewer_signal']
recommendations_df = streams_df.sort_values(['embedding_score', 'viewer_count'], ascending=False).head(50).copy()
recommendations_df.insert(0, 'rank', range(1, len(recommendations_df) + 1))
recommendations_df[['rank', 'user_name', 'game_name', 'title', 'viewer_count', 'metadata_similarity', 'embedding_score']]

,rank,user_name,game_name,title,viewer_count,metadata_similarity,embedding_score
0,1,Caedrel,League of Legends,🔴WORLDS 2026 KOI VS KC ELIMINATION🔴,137490,0.0,0.30000
1,2,Kamet0,League of Legends,KARMINE CORP 0 vs 0 MOVISTAR KOI | WORLDS 2026...,109134,0.0,0.29994
2,3,ibai,League of Legends,#WORLDS2026 | MKOI vs KC | EL PARTIDO DEL AÑO ...,66175,0.0,0.29988
3,4,otplol_,League of Legends,WORLDS - KC vs MKOI - POUR UNE PLACE EN QUART ...,36421,0.0,0.29982
4,5,Trymacs,Minecraft,100.000 TNT Exploion Craft Attack Rekord aufst...,31062,0.0,0.29976
5,6,Riot Games,League of Legends,KC vs MKOI | WORLDS 2026 | Swiss Stage Day 8,28350,0.0,0.29970
6,7,EDISON,Just Chatting,🎃КОНКУРС КОСТЮМОВ НА ХЕЛЛОУИН! [сейчас],27898,0.0,0.29964
7,8,Warframe,Warframe,"PMCF Quest Finale | Saturday, October 31 @ 12P...",26387,0.0,0.29958
8,9,ohnePixel,Just Chatting,🔴JAPAN STORY TIME🔴,22956,0.0,0.29952
9,10,afoninje,Dota 2,World Squad [0:0] CIS Squad | Bo1 | PARI UNIVE...,19762,0.0,0.29946


In [6]:
output_dir = data_dir
recommendations_df.to_json(output_dir / 'embedding_top_50.jsonl', orient='records', lines=True, date_format='iso')
streams_df[['id', 'user_id', 'game_id', 'game_name', 'metadata_similarity', 'embedding_score']].to_json(output_dir / 'stream_embedding_scores.jsonl', orient='records', lines=True, date_format='iso')
category_embeddings_df.to_json(output_dir / 'category_embeddings.jsonl', orient='records', lines=True)
channel_embeddings_df.to_json(output_dir / 'channel_embeddings.jsonl', orient='records', lines=True)
np.savez_compressed(output_dir / 'metadata_embeddings.npz', token_vectors=token_vectors, vocabulary=np.array(vocabulary, dtype=object), stream_vectors=stream_vectors)
print(f'Wrote embedding outputs to {output_dir}')

Wrote embedding outputs to /Users/edwynzhou/Documents/repos/TwitchEE/notebooks/data
